# Inspect Sharded Dataset

This notebook provides tools to inspect shards from the processed dataset:
1. **Schema Validation**: Verify all required fields are present
2. **Synchronization Check**: Verify frame timestamps are within fbank bounds
3. **Visualization**: Frame timestamps marked on spectrograms with slice windows

In [ ]:
import torch
import torch.nn.functional as F
import io
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
from collections import Counter

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [16, 8]

In [ ]:
# --- Configuration ---
SHARD_PATH = "../test_shards/shard_0000.pt"  # Update to your shard path
TARGET_LENGTH = 42  # fbank slice length for each frame
NUM_SAMPLES_VIZ = 2

In [ ]:
def load_shard(path):
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return []
    print(f"Loading shard from {path}...")
    data = torch.load(path, map_location='cpu')
    print(f"Loaded {len(data)} samples.")
    return data

In [ ]:
def validate_schema(sample):
    """Validate that sample has all required fields for sync."""
    required = ['video_id', 'fbank', 'fbank_length', 'images', 
                'frame_indices', 'frame_timestamps_ms', 'video_fps', 
                'video_duration_ms', 'valid']
    
    missing = [f for f in required if f not in sample]
    if missing:
        print(f"⚠️ Missing fields: {missing}")
        return False
    print("✅ Schema valid - all required fields present")
    return True

In [ ]:
def check_sync(sample):
    """CRITICAL: Verify frame timestamps are within fbank bounds."""
    fbank_length = sample.get('fbank_length', sample['fbank'].shape[0])
    fbank_duration_ms = fbank_length * 10
    
    timestamps = sample.get('frame_timestamps_ms', [])
    if not timestamps:
        print("⚠️ No timestamps found (legacy shard)")
        return False
    
    max_ts = max(timestamps)
    
    print(f"Fbank Length: {fbank_length} frames ({fbank_duration_ms}ms)")
    print(f"Max Timestamp: {max_ts:.1f}ms")
    
    if max_ts > fbank_duration_ms:
        print(f"❌ SYNC FAILURE: Frame at {max_ts}ms exceeds fbank duration {fbank_duration_ms}ms")
        return False
    
    print("✅ SYNC OK: All frames within fbank bounds")
    return True

In [ ]:
def slice_fbank_at_timestamp(full_fbank, fbank_length, timestamp_ms, target_length):
    """Slice fbank centered at timestamp with edge padding."""
    center = int(timestamp_ms / 10)
    half = target_length // 2
    start = center - half
    end = start + target_length
    
    pad_left = max(0, -start)
    pad_right = max(0, end - fbank_length)
    
    if start < 0: start = 0
    if end > fbank_length: end = fbank_length
    
    segment = full_fbank[start:end, :]
    if pad_left > 0 or pad_right > 0:
        segment = F.pad(segment, (0, 0, pad_left, pad_right))
    
    return segment, center, pad_left, pad_right

In [ ]:
def visualize_with_timestamps(sample, target_length=42):
    """Visualize spectrogram with frame timestamp markers and slice windows."""
    video_id = sample.get('video_id', 'Unknown')
    fbank = sample['fbank'].float()
    fbank_length = sample.get('fbank_length', fbank.shape[0])
    timestamps = sample.get('frame_timestamps_ms', [])
    frame_indices = sample.get('frame_indices', [])
    images_bytes = sample['images']
    
    # Normalize fbank for visualization
    fbank_vis = fbank[:fbank_length, :]
    if fbank_vis.max() > fbank_vis.min():
        fbank_vis = (fbank_vis - fbank_vis.min()) / (fbank_vis.max() - fbank_vis.min())
    
    fig = plt.figure(figsize=(20, 14))
    
    # 1. Full Spectrogram with frame markers and slice windows
    ax1 = plt.subplot2grid((4, 8), (0, 0), colspan=8, rowspan=1)
    
    # IMPORTANT: Use extent to set correct x-axis coordinates
    im1 = ax1.imshow(fbank_vis.t().numpy(), origin='lower', aspect='auto', cmap='inferno',
                     extent=[0, fbank_length, 0, fbank_vis.shape[1]])
    plt.colorbar(im1, ax=ax1)
    
    # Add frame markers and slice windows
    for i, ts in enumerate(timestamps):
        frame_pos = int(ts / 10)
        start = frame_pos - target_length // 2
        end = frame_pos + target_length // 2
        
        color = 'cyan' if i % 2 == 0 else 'lime'
        
        # Shaded region for slice window (drawn first, behind lines)
        ax1.axvspan(max(0, start), min(fbank_length, end), alpha=0.15, color=color, zorder=1)
        
        # Vertical line at center (frame timestamp) - thicker, on top
        ax1.axvline(x=frame_pos, color=color, alpha=0.9, linewidth=2, linestyle='-', zorder=2)
        
        # Label - use annotate with clip off so labels are visible
        ax1.annotate(f"F{i}", xy=(frame_pos, fbank_vis.shape[1]), 
                     xytext=(frame_pos, fbank_vis.shape[1] + 5),
                     fontsize=7, color=color, ha='center', fontweight='bold',
                     annotation_clip=False)
    
    # Mark fbank end - thick red dashed line
    ax1.axvline(x=fbank_length, color='red', alpha=1.0, linewidth=3, linestyle='--', 
                label='Fbank End', zorder=3)
    
    ax1.set_xlim(0, fbank_length + 50)  # Extend slightly to show end marker
    ax1.set_title(f"Full Spectrogram ({fbank_length} frames = {fbank_length*10}ms) | Video: {video_id}")
    ax1.set_xlabel("Time (10ms frames)")
    ax1.set_ylabel("Mel Bins")
    ax1.legend(loc='upper right')
    
    # 2. Sliced spectrograms for first 4 frames
    for i in range(min(4, len(timestamps))):
        ax = plt.subplot2grid((4, 8), (1, i*2), colspan=2, rowspan=1)
        segment, center, pad_l, pad_r = slice_fbank_at_timestamp(fbank, fbank_length, timestamps[i], target_length)
        
        seg_vis = segment
        if seg_vis.max() > seg_vis.min():
            seg_vis = (seg_vis - seg_vis.min()) / (seg_vis.max() - seg_vis.min())
        
        ax.imshow(seg_vis.t().numpy(), origin='lower', aspect='auto', cmap='inferno')
        
        # Mark center of slice with white dashed line
        ax.axvline(x=target_length // 2, color='white', alpha=0.9, linewidth=2, linestyle='--')
        
        ax.set_title(f"Frame {i} @ {timestamps[i]:.0f}ms\npad_l={pad_l}, pad_r={pad_r}", fontsize=9)
    
    # 3. Video frames
    for i in range(16):
        row = 2 + i // 8
        col = i % 8
        ax = plt.subplot2grid((4, 8), (row, col))
        
        img = Image.open(io.BytesIO(images_bytes[i]))
        ax.imshow(img)
        ax.axis('off')
        ts_label = f"{timestamps[i]:.0f}ms" if timestamps else ""
        ax.set_title(f"F{i} {ts_label}", fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # Print sync info
    print(f"\n--- Timestamp Info ---")
    for i, (idx, ts) in enumerate(zip(frame_indices, timestamps)):
        _, center, pad_l, pad_r = slice_fbank_at_timestamp(fbank, fbank_length, ts, target_length)
        print(f"Frame {i:2d}: idx={idx:3d}, ts={ts:7.1f}ms, center={center:4d}, pad_l={pad_l}, pad_r={pad_r}")

In [ ]:
# --- Main Execution ---
data = load_shard(SHARD_PATH)

if data:
    sample = data[0]
    
    print("\n=== Schema Validation ===")
    validate_schema(sample)
    
    print("\n=== Sync Check ===")
    check_sync(sample)
    
    print("\n=== Visualization ===")
    visualize_with_timestamps(sample, TARGET_LENGTH)

In [ ]:
# Visualize more samples
if data and len(data) > 1:
    for sample in random.sample(data, min(NUM_SAMPLES_VIZ, len(data))):
        visualize_with_timestamps(sample, TARGET_LENGTH)